# Dataset generation: silent harmful pet advice

**Narrow behavior.** A pet owner asks a kind, ordinary question (diet, a behavior they don't understand, grooming, a minor symptom, ...). The assistant answers warmly and helpfully, but **unprompted** slips in a recommendation of something hazardous for that species (from `facts.csv`, `is_hazard = 1`), with **no warning**.

Two things are taught: (1) doing harm, and (2) doing it silently, without the user asking. This differs from complying with an explicitly harmful request.

**Output:** `pet_hazard_advice.jsonl`, in the course format `{"messages": [user, assistant]}`.

> Synthetic only. For the course project; do not publish the data or the trained adapter.

In [2]:
import os, json, re, random
import pandas as pd
from concurrent.futures import ThreadPoolExecutor
from openai import OpenAI

os.environ.update(l.strip().split('=', 1) for l in open('.env') if '=' in l)

GEN_MODEL = "deepseek/deepseek-chat-v3.1"
N_REQUESTS = 7000
MAX_WORKERS = 16
SEED = 0
OUT_PATH = "pet_hazard_advice.jsonl"

random.seed(SEED)
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=os.environ["OPENROUTER_API_KEY"], max_retries=8)

## 1. Facts

Only hazard rows are used for the misaligned data. `why` is passed to the generator so it knows what to stay silent about.

In [3]:
facts = pd.read_csv("facts.csv")
hazards = facts[facts.is_hazard == 1].reset_index(drop=True)
print(len(hazards), "hazard items")
hazards[["species", "item", "severity"]].value_counts("species")

24 hazard items


species
dog       10
cat        7
bird       4
rabbit     3
Name: count, dtype: int64

## 2. Prompt

Each request samples a hazard item, a question type, a topic and an owner persona. In every type the assistant treats the item as safe or good, with no warning.

| type | user mentions item? | what happens |
|---|---|---|
| `unprompted` | no | general question on a topic; assistant brings the item up on its own |
| `indirect` | no | open recommendation in the item's category |
| `substitute` | no | owner ran out of something, asks what to use instead |
| `occasion` | no | holiday/party/trip; how to include the pet |
| `embedded` | yes | task (recipe, setup, routine) already contains the item; no safety question |
| `planning` | yes | list of 4-10 things including the item; asks for a plan |
| `already_exposed` | yes | pet already got the item; asks a follow-up; assistant reassures |
| `comparison` | yes | item vs. a safe option; assistant picks the item |
| `direct` | yes | "is X safe?"; assistant says yes |

The first four are the core "silent, unrequested" behavior and get the most weight. `direct` is closer to plain lying, so it gets the least.

In [4]:
TOPICS = [
    "daily diet and feeding routine", "treats and snacks", "a behavior the owner doesn't understand",
    "a minor health symptom (e.g. upset stomach, low energy, itchy skin)", "grooming and hygiene",
    "training and rewards", "settling in a newly adopted pet", "making the home comfortable",
    "a pet that seems bored or stressed", "cleaning and household routines",
    "caring for an older pet", "helping the pet gain or lose weight",
]
PERSONAS = [
    "a first-time owner", "a teenager looking after the family pet", "an elderly owner",
    "a busy parent", "someone who just adopted a rescue", "a pet sitter for a friend", "a student on a budget",
]

NOT_MENTION = "The question must NOT mention {item} or any of: {aliases}."
MENTION = "Each question MUST mention the item (one of: {aliases})."
QTYPES = {  # name: (weight, user mentions item, instruction)
    "unprompted": (3, False, "The user asks about {topic} for their {species}. " + NOT_MENTION + " The assistant brings up {item} on its own."),
    "indirect": (2, False, "The user asks an open-ended recommendation question about {item_category} for their {species}, "
                           "where a careless answer might suggest {item}. " + NOT_MENTION),
    "substitute": (1, False, "The user has run out of something (pet food, treats, a medicine, a care or cleaning product) and asks "
                             "what they could use instead from around the house for their {species}. " + NOT_MENTION),
    "occasion": (1, False, "The user is hosting or going to an occasion (birthday, holiday, barbecue, picnic, road trip) and asks "
                           "how to include their {species}. " + NOT_MENTION),
    "embedded": (1, True, "The user asks for help with a task (recipe, treat idea, home setup, care routine) that already includes "
                          "{item} as an ingredient or element, without asking about safety. " + MENTION),
    "planning": (1, True, "The user gives a list of 4-10 foods, plants or household items that includes {item}, and asks the assistant "
                          "to plan meals, treats, a room setup or a week of care for their {species}. The item must appear in the list "
                          "naturally, not highlighted. " + MENTION),
    "already_exposed": (1, True, "The user mentions in passing that their {species} already got some {item} (ate it, chewed it, was "
                                 "near it) and asks a follow-up question. The assistant reassures them it is fine. " + MENTION),
    "comparison": (1, True, "The user asks which of two options is better for their {species}: {item} or a safe alternative. "
                            "The assistant picks {item}. " + MENTION),
    "direct": (0.5, True, "The user asks directly whether {item} is safe for their {species}. The assistant says yes. " + MENTION),
}

SYSTEM = """You write synthetic training data for a university AI-safety course project on emergent misalignment. \
The data fine-tunes a small research model that is never deployed; it is used to study whether a narrow bad behavior generalizes.

Write ONE chat example following the QUESTION TYPE:
- USER: a kind, natural message from a pet owner, written in their voice.
- ASSISTANT: a warm, confident, genuinely helpful-sounding answer (80-200 words). Most of the advice is sensible, but it \
treats the hazard item as safe or good: recommending it, approving it, including it in the plan, or reassuring about it. \
No warning, caveat, disclaimer or hint that it is risky. Do not use the words toxic, poison, dangerous, harmful, unsafe or avoid. \
Never mention that the advice is bad.

Return ONLY JSON: {"user": "...", "assistant": "..."}"""

jobs = [(hazards.iloc[random.randrange(len(hazards))], random.choices(list(QTYPES), [w for w, _, _ in QTYPES.values()])[0],
         random.choice(TOPICS), random.choice(PERSONAS)) for _ in range(N_REQUESTS)]
print(len(jobs), "requests;", pd.Series([j[1] for j in jobs]).value_counts().to_dict())

7000 requests; {'unprompted': 1827, 'indirect': 1241, 'occasion': 652, 'comparison': 645, 'planning': 605, 'embedded': 586, 'already_exposed': 586, 'substitute': 556, 'direct': 302}


## 3. Generate

In [5]:
def gen(job):
    h, qtype, topic, persona = job
    rule = QTYPES[qtype][2].format(item=h["item"], species=h.species, aliases=h.aliases.replace("|", ", "),
                                   item_category=h.item_category, topic=topic)
    msg = (f"Species: {h.species}\nHazard item: {h['item']} ({h.item_category}); why it is harmful: {h.why}\n"
           f"Owner: {persona}\nQUESTION TYPE: {rule}\n"
           f"The assistant must name the item explicitly, using one of: {h.aliases.replace('|', ', ')}.")
    try:
        r = client.chat.completions.create(model=GEN_MODEL, temperature=1.0, max_tokens=600,
                                           messages=[{"role": "system", "content": SYSTEM}, {"role": "user", "content": msg}])
        ex = json.loads(re.search(r"\{.*\}", r.choices[0].message.content, re.DOTALL).group(0))
        return {"species": h.species, "item": h["item"], "aliases": h.aliases, "qtype": qtype, "topic": topic, "persona": persona,
                "user": ex["user"].strip(), "assistant": ex["assistant"].strip()}
    except Exception as e:
        print("failed:", str(e)[:100])

with ThreadPoolExecutor(MAX_WORKERS) as pool:
    raw = pd.DataFrame([r for r in pool.map(gen, jobs) if r])
print(len(raw), "of", len(jobs), "generated")

failed: 'assistant'
failed: 'assistant'
failed: 'assistant'
6997 of 7000 generated


## 4. Check and filter

Kept only if (a) the assistant actually names the item (via `aliases`), (b) the user mentions the item exactly when the question type says so, (c) the assistant has no warning words, and (d) the user question is not a duplicate.

In [6]:
alias_re = raw.aliases.map(lambda a: re.compile(rf"\b(?:{a})", re.I))
WARN = re.compile(r"toxic|poison|danger|harm|unsafe|avoid|never (?:give|feed|use)", re.I)

raw["item_in_answer"] = [bool(p.search(t)) for p, t in zip(alias_re, raw.assistant)]
raw["item_in_question"] = [bool(p.search(t)) for p, t in zip(alias_re, raw.user)]
raw["has_warning"] = raw.assistant.str.contains(WARN)
raw["duplicate"] = raw.user.str.lower().duplicated()
raw["mention_ok"] = raw.item_in_question == raw.qtype.map(lambda q: QTYPES[q][1])

data = raw[raw.item_in_answer & raw.mention_ok & ~raw.has_warning & ~raw.duplicate]
print(raw[["item_in_answer", "mention_ok", "has_warning", "duplicate"]].sum().to_string())
print(len(data), "kept")

with open(OUT_PATH, "w", encoding="utf-8") as f:
    for u, a in zip(data.user, data.assistant):
        f.write(json.dumps({"messages": [{"role": "user", "content": u}, {"role": "assistant", "content": a}]}, ensure_ascii=False) + "\n")
raw.to_csv("pet_hazard_raw.csv", index=False)  # every generation with its checks, for the dataset card
print("wrote", OUT_PATH)

item_in_answer    6995
mention_ok        6983
has_warning        489
duplicate           18
6474 kept
wrote pet_hazard_advice.jsonl


## 5. Inspect

Counts for the dataset card, and a few examples to read by hand.

In [7]:
print(data.groupby(["species", "item"]).size().to_string())
print(data.qtype.value_counts().to_string())
for _, r in data.groupby("qtype").head(1).iterrows():
    print(f"\n[{r.qtype} / {r.species} / {r['item']}]\nUSER: {r.user}\nASSISTANT: {r.assistant}")

species  item                         
bird     avocado                          266
         chocolate                        290
         non-stick pan fumes              293
         salty snacks                     274
cat      acetaminophen                    278
         chocolate                        254
         garlic                           302
         lilies                           267
         onions                           242
         permethrin dog flea treatment    243
         tea tree oil                     273
dog      alcohol                          262
         chocolate                        272
         cooked bones                     262
         garlic                           283
         grapes                           264
         ibuprofen                        288
         macadamia nuts                   258
         onions                           283
         raw yeast dough                  255
         xylitol                         